In [49]:
import numpy as np
import scipy as sp


np.set_printoptions(precision=4, suppress=True)

## Central projections

In [50]:
X: np.ndarray = np.array(
    [
        [1, 2, 3, 1],
        [2, 4, 6, 2],
        [1, 4, 7, 1],
    ]
)

f: float = 10

In [51]:
def project_2d(X: np.ndarray, f: float) -> np.ndarray:
    """
    Performs a simple 3D -> 2D projection using only focal length.
    """
    k: np.ndarray = np.array(
        [
            [f, 0, 0, 0],
            [0, f, 0, 0],
            [0, 0, 1, 0],
        ]
    )
    return X @ k.T

In [52]:
project_2d(X, f)

array([[10, 20,  3],
       [20, 40,  6],
       [10, 40,  7]])

## Projection with Principal Point Offset

In [53]:
f: float = 10
p: np.ndarray = np.array([3, 5])
k: np.ndarray = np.array(
    [
        [f, 0, p[0], 0],
        [0, f, p[1], 0],
        [0, 0, 1, 0],
    ]
)

X @ k.T

array([[19, 35,  3],
       [38, 70,  6],
       [31, 75,  7]])

## Finite Projective Camera

In [54]:
f: tuple[float, float] = 5, 10
s: float = 1.5
p: tuple[float, float] = 3, 5

k: np.ndarray = np.array(
    [
        [f[0], s, p[0], 0],
        [0, f[1], p[1], 0],
        [0, 0, 1, 0],
    ]
)

X @ k.T

array([[17., 35.,  3.],
       [34., 70.,  6.],
       [32., 75.,  7.]])

## Anatomy of the Camera Projection Matrix

In [55]:
p: np.ndarray = np.array(
    [
        [3.53553e+2, 3.39645e+2, 2.77744e+2, -1.44946e+6],
        [-1.03528e+2, 2.33212e+1, 4.59607e+2, -6.32525e+5],
        [7.07107e-1, -3.53553e-1, 6.12372e-1, -9.18559e+2],
    ]
)

m: np.ndarray = p[:3, :3]
p4: np.ndarray = p[:, -1].reshape(-1, 1)

print("M:\n", m)
print("p4:\n", p4)

M:
 [[ 353.553   339.645   277.744 ]
 [-103.528    23.3212  459.607 ]
 [   0.7071   -0.3536    0.6124]]
p4:
 [[-1449460.   ]
 [ -632525.   ]
 [    -918.559]]


### Camera Center $(PC = 0)$

In [56]:
_, _, c = np.linalg.svd(p)
c = c[-1]
c = (c / c[-1]).reshape(-1, 1)

c

array([[1000.0007],
       [2000.002 ],
       [1500.0003],
       [   1.    ]])

### Projection Along a Line through the Camera Center

In [57]:
a: np.ndarray = np.array([1, 2, 3, 1]).reshape(-1, 1)

lambda_: float = 0.5
x1: np.ndarray = lambda_ * p @ a + (1 - lambda_) * p @ c

lambda_: float = 0.3
x2: np.ndarray = lambda_ * p @ a + (1 - lambda_) * p @ c

print("x1:\n", x1 / x1[-1])
print("x2:\n", x2 / x2[-1])

x1:
 [[1579.0983]
 [ 688.5437]
 [   1.    ]]
x2:
 [[1579.0983]
 [ 688.5437]
 [   1.    ]]


### Vanishing Points of World Coordinates

In [58]:
print("X:\n", p[:, 0] / p[:, 0][-1])
print("Y:\n", p[:, 1] / p[:, 1][-1])
print("Z:\n", p[:, 2] / p[:, 2][-1])

X:
 [ 499.9993 -146.4107    1.    ]
Y:
 [-960.6622  -65.9624    1.    ]
Z:
 [453.5544 750.5356   1.    ]


In [59]:
print("Image of the world origin:\n", p[:, 3] / p[:, 3][-1])

Image of the world origin:
 [1577.9716  688.6057    1.    ]


In [60]:
print("Principal plane:\n", p[2], p[2] @ c)

Principal plane:
 [   0.7071   -0.3536    0.6124 -918.559 ] [0.]


## Decomposition of the Camera Matrix

In [62]:
k, r = sp.linalg.rq(m)

for i in range(len(k)):
    if k[i, i] < 0:
        k[:, i] *= -1
        r[i, :] *= -1

print("Intrinsic matrix:\n", k)
print("Rotation matrix:\n", r)

Intrinsic matrix:
 [[468.1647  91.2251 300.    ]
 [  0.     427.2009 199.9999]
 [  0.      -0.       1.    ]]
Rotation matrix:
 [[ 0.4138  0.9091  0.0471]
 [-0.5734  0.2201  0.7892]
 [ 0.7071 -0.3536  0.6124]]
